## Validation design and candidate models

**Validation design.** All evaluation uses rolling-origin (expanding-window) validation inside the 110-point training window. The first origin has 48 points of history (4 hours). Each later origin moves forward by 6 points (30 minutes) and the history grows. From each origin the model forecasts the next 24 points (2 hours), the same horizon as the final task, using only data up to that origin. This gives 7 origins (168 forecast points) and mimics production: forecast the next two hours from everything seen so far. The windows overlap (6-point step, 24-point horizon), so the 7 origins are not independent and small differences should not be over-read. Peak periods are the points at or above the 90th percentile of the training series.

**Candidates.**
- `last_value`: repeats the last observed value.
- `train_mean`: the mean of all history up to the origin.
- `moving_avg_k`: the mean of the last k points (k = 3, 6, 12, i.e. 15, 30, 60 minutes).
- `damped_persistence`: mean + φ^h × (last value − mean), where φ is the lag-1 autocorrelation of the history.
- `seasonal_naive_12`: repeats the last hour. It is a reference only, because about 11 hours of data cannot establish a seasonal period.

In [9]:
from functools import partial
import numpy as np
import pandas as pd

from cwfc.data import load_cpu, build_targets, split_train_holdout
from cwfc.baselines import (persistence_forecast, moving_average_forecast,
                            train_mean_forecast, seasonal_naive_forecast)
from cwfc.models import damped_persistence_forecast
from cwfc.validation import rolling_origin_forecasts
from cwfc.metrics import mae, rmse, relative_improvement, subset_metrics

cpu = load_cpu("../challenge_data/vm_cpu_readings.csv.gz")
targets = build_targets(cpu)
train, _ = split_train_holdout(targets)   # the holdout is deliberately discarded here
print(len(train))                          # 110

110


In [10]:
candidates = {
    "last_value": persistence_forecast,
    "train_mean": train_mean_forecast,
    "moving_avg_3": partial(moving_average_forecast, window=3),
    "moving_avg_6": partial(moving_average_forecast, window=6),
    "moving_avg_12": partial(moving_average_forecast, window=12),
    "damped_persistence": damped_persistence_forecast,
    "seasonal_naive_12 (reference)": partial(seasonal_naive_forecast, season_length=12),
}

def run_candidates(target):
    return {name: rolling_origin_forecasts(train, target, fn)
            for name, fn in candidates.items()}

## Target 1: `active_vm_count`

In [11]:
target_a = "active_vm_count"
results_a = run_candidates(target_a)

table_a = pd.DataFrame([
    {"model": n, "MAE": mae(r.actual, r.pred), "RMSE": rmse(r.actual, r.pred)}
    for n, r in results_a.items()
])
print(table_a.round(4))

                           model  MAE  RMSE
0                     last_value  0.0   0.0
1                     train_mean  0.0   0.0
2                   moving_avg_3  0.0   0.0
3                   moving_avg_6  0.0   0.0
4                  moving_avg_12  0.0   0.0
5             damped_persistence  0.0   0.0
6  seasonal_naive_12 (reference)  0.0   0.0


**What this shows:** `active_vm_count` is constant at 10,000, so every method has zero MAE and RMSE. Relative improvement and peak metrics are undefined here (the baseline error is zero and every point equals the 90th percentile). The simplest method, `last_value`, will be used for this target.

## Target 2: `aggregate_cpu_utilization`

In [12]:
target_b = "aggregate_cpu_utilization"
results_b = run_candidates(target_b)

peak_threshold = train[target_b].quantile(0.90)   # recomputed from train, never typed in
ref_last = results_b["last_value"]
ref_mean = results_b["train_mean"]

rows = []
for name, r in results_b.items():
    peak = subset_metrics(r.actual, r.pred, r.actual >= peak_threshold)
    rows.append({
        "model": name,
        "MAE": mae(r.actual, r.pred),
        "RMSE": rmse(r.actual, r.pred),
        "vs_last_value": relative_improvement(r.actual, r.pred, ref_last.pred),
        "vs_train_mean": relative_improvement(r.actual, r.pred, ref_mean.pred),
        "peak_MAE": peak["mae"],
    })
table_b = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
print(table_b.round(4))

                           model     MAE    RMSE  vs_last_value  \
0             damped_persistence  0.2025  0.2419         0.2372   
1                     train_mean  0.2090  0.2468         0.2128   
2                   moving_avg_3  0.2439  0.2953         0.0815   
3                   moving_avg_6  0.2532  0.3051         0.0463   
4                     last_value  0.2655  0.3174         0.0000   
5                  moving_avg_12  0.2688  0.3121        -0.0124   
6  seasonal_naive_12 (reference)  0.2809  0.3293        -0.0580   

   vs_train_mean  peak_MAE  
0         0.0310    0.3961  
1         0.0000    0.4157  
2        -0.1667    0.3369  
3        -0.2115    0.3485  
4        -0.2703    0.2455  
5        -0.2860    0.3790  
6        -0.3440    0.3985  


**What this shows:** `damped_persistence` has the lowest MAE (0.2025) and RMSE (0.2419). It is 3.1% better than `train_mean` in MAE (0.2090) and 23.7% better than `last_value` (0.2655), so the gap to `train_mean` is small. `last_value` is best on peak points (peak MAE 0.246 versus 0.396 for `damped_persistence`) but weaker overall. `seasonal_naive_12` is worse than `last_value`, so there is no evidence of a repeating one-hour pattern. The typical errors (0.2 to 0.4 points) are about as large as the hump itself (about 0.5 points above average), so no baseline follows the peak well.

## Diagnostics (target 2 only): is one model best everywhere?

In [13]:
contenders = [n for n in results_b if "reference" not in n]
abs_err = {n: (r.actual - r.pred).abs() for n, r in results_b.items()}

per_origin = pd.DataFrame({
    n: abs_err[n].groupby(results_b[n]["origin"]).mean() for n in results_b
}).round(3)

windows = ref_last.groupby("origin")["timestamp"].agg(window_start="min", window_end="max")
print(per_origin.join(windows))

print("origins won (reference excluded):")
print(per_origin[contenders].idxmin(axis=1).value_counts())

        last_value  train_mean  moving_avg_3  moving_avg_6  moving_avg_12  \
origin                                                                      
0            0.298       0.297         0.310         0.333          0.316   
1            0.160       0.302         0.279         0.285          0.312   
2            0.305       0.191         0.313         0.289          0.179   
3            0.464       0.154         0.333         0.328          0.362   
4            0.257       0.160         0.271         0.332          0.374   
5            0.257       0.165         0.076         0.096          0.221   
6            0.118       0.195         0.125         0.109          0.118   

        damped_persistence  seasonal_naive_12 (reference)  window_start  \
origin                                                                    
0                    0.297                          0.309         14400   
1                    0.302                          0.316         16200   
2     

**What this shows:** No model wins everywhere: five different models are best at least once, and `train_mean` and `damped_persistence` each lead in 2 of 7 origins (origin 0 is a tie at 0.297). Error depends mostly on where the window falls relative to the hump. `last_value` is worst at origin 3 (0.464; window 19,800 to 26,700 s, starting right after a peak reading, then the level falls) and best at origin 1 (0.160; the window starts just after the climb at 15,900 s, so repeating the elevated value fits). Operationally, the forecast is least reliable when the load is about to change direction.

In [14]:
per_step = pd.DataFrame({
    n: abs_err[n].groupby(results_b[n]["step"]).mean() for n in results_b
}).round(3)
print(per_step.loc[[1, 3, 6, 12, 18, 24]])   # 5 min, 15 min, 30 min, 1 h, 1.5 h, 2 h

      last_value  train_mean  moving_avg_3  moving_avg_6  moving_avg_12  \
step                                                                      
1          0.121       0.166         0.090         0.097          0.124   
3          0.117       0.225         0.157         0.154          0.186   
6          0.150       0.224         0.146         0.162          0.222   
12         0.242       0.219         0.250         0.246          0.250   
18         0.267       0.167         0.236         0.254          0.256   
24         0.257       0.132         0.252         0.248          0.238   

      damped_persistence  seasonal_naive_12 (reference)  
step                                                     
1                  0.103                          0.212  
3                  0.202                          0.250  
6                  0.216                          0.252  
12                 0.218                          0.242  
18                 0.167                          0

**What this shows:** Recent-value methods are best at short steps and the long-run mean is best at the end (step 24: 0.132 for `train_mean` versus 0.257 for `last_value`); the curves cross around step 12 (1 hour). `damped_persistence` is good at step 1 and at steps 18 to 24, but at steps 3 to 6 it is clearly worse than `last_value` and `moving_avg_3` (for example 0.202 versus 0.117 at step 3). Its AR(1) decay is faster than the real memory of this series: with φ near 0.67 the weight on the last value is about 0.09 after 30 minutes, while the observed autocorrelation at 30 minutes is still 0.48. A model that can capture slower decay is worth testing.

In [15]:
peak_rows = ref_last[ref_last.actual >= peak_threshold]
print("peak points:", len(peak_rows), "| distinct timestamps:", peak_rows["timestamp"].nunique())
print("peak threshold:", round(peak_threshold, 3))

peak points: 30 | distinct timestamps: 11
peak threshold: 9.668


**What this shows:** The 30 peak points come from only 11 distinct timestamps, because overlapping windows count the same timestamp several times. Peak results are therefore weak evidence.

**Baseline decision**

- Reference baselines: `last_value` (standard) and `train_mean` (strongest simple baseline).
- Best statistical candidate so far: `damped_persistence`, MAE 0.2025 and RMSE 0.2419. It improves on `train_mean` by only 3.1% (MAE) and 2.0% (RMSE) and has the lowest error in just 2 to 3 of 7 origins, so the gap is within noise.
- Weak spots: steps 3 to 12 (where `last_value` and short moving averages do better) and peak periods (which rest on 11 distinct timestamps).
- The bar for any later model is the lowest rolling-origin MAE so far (0.2025), not `last_value`.
- For `active_vm_count`, `last_value` is used (zero error).
- Next (notebook 04): test one ML model (Ridge regression with lag and rolling-mean features). It is kept only if it beats `damped_persistence` on rolling-origin MAE without a worse peak MAE. This rule is fixed before running it.